# RW-ASR: Risk-Weighted Attack Success Rate on KITTI

Attack success rate (ASR) counts every misclassification the same. RW-ASR weights
each failure by how much it raises the chance of a fatal crash.

$$\text{RW-ASR} = \frac{\sum_i w_i \cdot \mathbf{1}[\text{fail}_i]}{\sum_i w_i}
\qquad
w_i = s(c_i)\cdot\left[P_{fatal}(v_{fail}) - P_{fatal}(v_{ok})\right]$$

The weight only uses two things from the KITTI labels: object class and distance (z).

Run on Colab with a T4 GPU (Runtime > Change runtime type).

## 1. Setup

In [ ]:
import os, math, json, random, time, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

In [ ]:
SEED = 1337

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()

FIGDIR = Path("figures"); FIGDIR.mkdir(exist_ok=True)
RESDIR = Path("results"); RESDIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

def savefig(name):
    plt.savefig(FIGDIR / f"{name}.png")

## 2. Config

| Constant | Value | Source |
|---|---|---|
| Fatality curve | logistic fit | Tefft (2013), Accident Analysis & Prevention 50 |
| Severity | ped 7, cyclist 2, car 1 | NHTSA Traffic Safety Facts 2023 (lethality index) |
| Deceleration | 6.0 m/s² | FMVSS 127 |
| Ego speed | 35 mph | assumed (KITTI labels have no speed), swept |
| Failure delay | 0.4 s | assumed, swept |

In [ ]:
CFG = dict(
    ego_speed_mph = 35.0,  # assumed
    decel         = 6.0,   # m/s^2
    react_ok      = 0.5,   # s, reaction time when the label is right
    delay_fail    = 0.4,   # s, extra delay from a wrong label (assumed)
)

SEVERITY = {
    "car": 1.0,
    "pedestrian": 7.0,
    "cyclist": 2.0,
    "non-road object": 1.0,
    "road scene": 0.0,  # background, nobody gets hit
}

CLASSES = ["car", "pedestrian", "cyclist", "road scene", "non-road object"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# KITTI types -> 5 classes (DontCare is dropped)
KITTI_MAP = {
    "Car": "car", "Van": "car", "Truck": "car",
    "Pedestrian": "pedestrian", "Person_sitting": "pedestrian",
    "Cyclist": "cyclist",
    "Tram": "non-road object", "Misc": "non-road object",
}

IMG_SIZE = 64
BATCH = 128
EPOCHS = 12
LR = 1e-3
MAX_PER_CLASS = 6000

# images are normalized to [-1, 1], so eps here is 2x the [0, 1] value
EPSILONS = [0.01, 0.03, 0.05, 0.10]                        # first sweep (mostly saturated)
EPSILONS_FINE = [0.001, 0.002, 0.004, 0.006, 0.008, 0.012]  # main sweep

def eps_over_255(e):
    return e / 2 * 255

## 3. Data

Put the KITTI zips in Google Drive at `MyDrive/KITTI/`:
```
data_object_label_2.zip
data_object_image_2.zip
```
They get copied to local disk first because reading thousands of files straight from Drive is very slow.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/KITTI")
LOCAL_DIR = Path("/content/kitti")
CACHE_DIR = DRIVE_DIR / "cache"   # crops + model weights saved here
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
LABEL_ZIP = DRIVE_DIR / "data_object_label_2.zip"
IMAGE_ZIP = DRIVE_DIR / "data_object_image_2.zip"

def stage(zip_path, dest=LOCAL_DIR, marker=None):
    dest.mkdir(parents=True, exist_ok=True)
    if marker and (dest / marker).exists():
        return
    local_zip = Path("/content") / zip_path.name
    if not local_zip.exists():
        shutil.copy(zip_path, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(dest)

stage(LABEL_ZIP, marker="training/label_2")

# images only needed the first time (to build the crop cache)
if not (CACHE_DIR / "crops.pt").exists():
    stage(IMAGE_ZIP, marker="training/image_2")

LABEL_DIR = LOCAL_DIR / "training" / "label_2"
IMAGE_DIR = LOCAL_DIR / "training" / "image_2"
print("label files:", len(list(LABEL_DIR.glob("*.txt"))))

### Parse labels

KITTI labels have 15 columns per object. Column 13 is the forward distance z (meters), which the risk weight needs.

In [ ]:
def parse_kitti_labels(label_dir):
    rows = []
    for f in sorted(Path(label_dir).glob("*.txt")):
        for line in f.read_text().strip().splitlines():
            p = line.split()
            cls = KITTI_MAP.get(p[0]) if p else None
            if cls is None:
                continue
            rows.append(dict(
                img_id=f.stem, raw_type=p[0], cls=cls,
                truncated=float(p[1]), occluded=int(p[2]),
                x1=float(p[4]), y1=float(p[5]), x2=float(p[6]), y2=float(p[7]),
                z=float(p[13]),
            ))
    df = pd.DataFrame(rows)
    df["bbox_h"] = df.y2 - df.y1
    return df

df = parse_kitti_labels(LABEL_DIR)
print(df.shape)
print(df.cls.value_counts())

## 4. EDA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

vc = df.cls.value_counts().reindex(["car", "pedestrian", "cyclist", "non-road object"])
ax[0].bar(vc.index, vc.values, color="#1f3864")
ax[0].set_title("Object class distribution (KITTI, simplified)")
ax[0].set_ylabel("labelled instances")
ax[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(vc.values):
    ax[0].text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=9)

for c, col in [("car", "#1f3864"), ("pedestrian", "#c0392b"), ("cyclist", "#d68910")]:
    ax[1].hist(df[df.cls == c].z, bins=np.arange(0, 81, 2.5), alpha=0.55, label=c, color=col)
ax[1].set_xlabel("forward distance z (m)")
ax[1].set_ylabel("count")
ax[1].set_title("Distance distribution by class")
ax[1].legend()

plt.tight_layout(); savefig("fig2_eda"); plt.show()

## 5. Risk model

1. **Braking:** impact speed after reacting at time t to an object at distance z:
   $v_{impact} = \sqrt{\max(0,\ v_0^2 - 2a(z - v_0 t))}$
2. **Fatality curve:** logistic fit to Tefft (2013): 10% at 24.1 mph, 25% at 32.5, 50% at 40.6, 75% at 48.0, 90% at 54.6.
3. **Weight:** run braking twice (normal reaction vs. reaction + delay) and take the difference in P(fatal), times class severity.

In [ ]:
from scipy.optimize import curve_fit

MPH = 0.44704  # m/s per mph

TEFFT_V = np.array([24.1, 32.5, 40.6, 48.0, 54.6])
TEFFT_P = np.array([0.10, 0.25, 0.50, 0.75, 0.90])

def _logistic(v, b1, b0):
    return 1.0 / (1.0 + np.exp(-(b1 * v + b0)))

TEFFT_B1, TEFFT_B0 = curve_fit(_logistic, TEFFT_V, TEFFT_P, p0=[0.13, -5.3])[0]
print("max fit error:", round(float(np.max(np.abs(_logistic(TEFFT_V, TEFFT_B1, TEFFT_B0) - TEFFT_P))), 4))

def p_fatal(v_mph):
    return _logistic(np.asarray(v_mph, dtype=float), TEFFT_B1, TEFFT_B0)

def impact_speed_mph(z, react_s, v0_mph, decel):
    v0 = v0_mph * MPH
    d = np.maximum(0.0, np.asarray(z, dtype=float) - v0 * react_s)
    return np.sqrt(np.maximum(0.0, v0 * v0 - 2.0 * decel * d)) / MPH

def risk_weight(cls, z, cfg=None, severity=None):
    # extra P(fatal) caused by misclassifying this object
    cfg = CFG if cfg is None else cfg
    sev = (SEVERITY if severity is None else severity).get(cls, 1.0)
    z = np.asarray(z, dtype=float)
    if sev == 0.0:
        return np.zeros_like(z)
    v_ok = impact_speed_mph(z, cfg["react_ok"], cfg["ego_speed_mph"], cfg["decel"])
    v_bad = impact_speed_mph(z, cfg["react_ok"] + cfg["delay_fail"], cfg["ego_speed_mph"], cfg["decel"])
    w = sev * np.maximum(0.0, p_fatal(v_bad) - p_fatal(v_ok))
    return np.where(np.isfinite(z) & (z > 0), w, 0.0)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

vv = np.linspace(0, 65, 300)
ax[0].plot(vv, p_fatal(vv), color="#1f3864", lw=2, label="logistic fit")
ax[0].scatter(TEFFT_V, TEFFT_P, color="#c0392b", zorder=5, label="Tefft (2013) published")
ax[0].set_xlabel("impact speed (mph)"); ax[0].set_ylabel("P(fatality)")
ax[0].set_title("Pedestrian fatality risk vs impact speed"); ax[0].legend()

zz = np.linspace(1, 60, 400)
for c, col in [("pedestrian", "#c0392b"), ("cyclist", "#d68910"), ("car", "#1f3864")]:
    ax[1].plot(zz, risk_weight(c, zz), lw=2, color=col, label=c)
ax[1].set_xlabel("object distance z (m)")
ax[1].set_ylabel("risk weight w (added P(fatal))")
ax[1].set_title(f"Weight vs distance @ {CFG['ego_speed_mph']:.0f} mph")
ax[1].legend()

plt.tight_layout(); savefig("fig3_risk_model"); plt.show()

pk = zz[int(np.argmax(risk_weight("pedestrian", zz)))]
print(f"pedestrian weight peaks at {pk:.1f} m")

In [ ]:
df["w"] = [float(risk_weight(c, z)) for c, z in zip(df.cls, df.z)]
print("share of total risk weight by class:")
print((df.groupby("cls")["w"].sum() / df.w.sum()).sort_values(ascending=False).round(3))

## 6. Crop dataset

One 64x64 crop per labeled object, plus random background ("road scene") patches.
Crops are saved to Drive the first time so later runs skip the 12 GB image zip.
Train/test split is by image id so no photo ends up in both.

In [ ]:
NORM_MEAN, NORM_STD = 0.5, 0.5   # -> [-1, 1]
CACHE_FILE = CACHE_DIR / "crops.pt"

def build_records(df, seed=SEED):
    rng = random.Random(seed)
    recs = []
    for r in df.itertuples():
        if (r.x2 - r.x1) < 8 or (r.y2 - r.y1) < 8:
            continue
        recs.append(dict(img_id=r.img_id, x1=r.x1, y1=r.y1, x2=r.x2, y2=r.y2,
                         cls=r.cls, label=CLASS_TO_IDX[r.cls], w=float(r.w), z=float(r.z)))

    # one background patch per image that doesn't overlap any box
    boxes = df.groupby("img_id")[["x1", "y1", "x2", "y2"]].apply(lambda g: g.values.tolist()).to_dict()
    for img_id, bl in boxes.items():
        for _ in range(12):
            s = rng.randint(60, 140)
            x1 = rng.randint(0, 1100); y1 = rng.randint(200, 340)
            if all(x1 + s < b[0] or x1 > b[2] or y1 + s < b[1] or y1 > b[3] for b in bl):
                recs.append(dict(img_id=img_id, x1=x1, y1=y1, x2=x1 + s, y2=y1 + s,
                                 cls="road scene", label=CLASS_TO_IDX["road scene"],
                                 w=0.0, z=float("inf")))
                break
    return recs

def cap_per_class(recs, cap=MAX_PER_CLASS, seed=SEED):
    rng = random.Random(seed)
    by = {}
    for r in recs:
        by.setdefault(r["cls"], []).append(r)
    out = []
    for lst in by.values():
        rng.shuffle(lst); out.extend(lst[:cap])
    rng.shuffle(out)
    return out

def extract_crops(recs, image_dir, size=IMG_SIZE):
    X = torch.zeros(len(recs), 3, size, size, dtype=torch.uint8)
    by_img = {}
    for i, r in enumerate(recs):
        by_img.setdefault(r["img_id"], []).append(i)
    for img_id, idxs in by_img.items():
        img = Image.open(Path(image_dir) / f"{img_id}.png").convert("RGB")
        for i in idxs:
            r = recs[i]
            c = img.crop((r["x1"], r["y1"], r["x2"], r["y2"])).resize((size, size))
            X[i] = torch.from_numpy(np.array(c, dtype=np.uint8)).permute(2, 0, 1)
    return X

if CACHE_FILE.exists():
    blob = torch.load(CACHE_FILE, weights_only=False)  # our own file, contains a DataFrame
    X_all, meta = blob["X"], blob["meta"]
else:
    records = cap_per_class(build_records(df))
    X_all = extract_crops(records, IMAGE_DIR)
    meta = pd.DataFrame(records)
    torch.save({"X": X_all, "meta": meta}, CACHE_FILE)

print(meta.cls.value_counts())

In [ ]:
class CropDataset(Dataset):
    def __init__(self, X, meta, train=True):
        self.X = X
        self.y = torch.tensor(meta.label.values, dtype=torch.long)
        self.w = torch.tensor(meta.w.values, dtype=torch.float32)
        self.train = train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = self.X[i].float().div(255.0)
        if self.train and random.random() < 0.5:
            x = torch.flip(x, dims=[2])  # random horizontal flip
        return (x - NORM_MEAN) / NORM_STD, self.y[i], self.w[i]

def split_by_image(meta, test_frac=0.2, seed=SEED):
    rng = random.Random(seed)
    ids = sorted(meta.img_id.unique())
    rng.shuffle(ids)
    test_ids = set(ids[:int(len(ids) * test_frac)])
    is_test = meta.img_id.isin(test_ids).values
    return ~is_test, is_test

tr_mask, te_mask = split_by_image(meta)
train_meta = meta[tr_mask].reset_index(drop=True)
test_meta = meta[te_mask].reset_index(drop=True)
assert not set(train_meta.img_id) & set(test_meta.img_id)

train_dl = DataLoader(CropDataset(X_all[tr_mask], train_meta, True), batch_size=BATCH,
                      shuffle=True, num_workers=2, pin_memory=True)
test_dl = DataLoader(CropDataset(X_all[te_mask], test_meta, False), batch_size=BATCH,
                     shuffle=False, num_workers=2, pin_memory=True)

test_cls = test_meta.cls.values
test_z = test_meta.z.values.astype(float)
print(f"train {tr_mask.sum()} | test {te_mask.sum()}")

## 7. Models

Baseline CNN and ResNet-18 (ImageNet pretrained). Tree/SVM models weren't used because
FGSM and PGD need input gradients.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, n_classes=len(CLASSES)):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                 nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                 nn.MaxPool2d(2))
        self.features = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.features(x))

def make_resnet18(n_classes=len(CLASSES)):
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

In [ ]:
@torch.no_grad()
def evaluate(model, dl):
    model.eval(); corr = n = 0
    for x, y, _ in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        corr += (model(x).argmax(1) == y).sum().item(); n += y.size(0)
    return corr / n

def train_model(model, epochs=EPOCHS, lr=LR, tag="model"):
    set_seed()
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    for ep in range(epochs):
        model.train()
        for x, y, _ in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            crit(model(x), y).backward()
            opt.step()
        sched.step()
        print(f"[{tag}] epoch {ep + 1}  test acc {evaluate(model, test_dl):.4f}")
    return model

cnn = train_model(BaselineCNN(), tag="cnn")
torch.save(cnn.state_dict(), CACHE_DIR / "cnn.pt")

rn18 = train_model(make_resnet18(), lr=3e-4, tag="resnet18")
torch.save(rn18.state_dict(), CACHE_DIR / "resnet18.pt")

MODELS = {"cnn": cnn, "resnet18": rn18}

## 8. Attacks

FGSM and PGD (10 steps, step size 2.5·eps/steps so PGD can reach the edge of the eps-ball).

In [ ]:
def fgsm(model, x, y, eps):
    x = x.clone().detach().requires_grad_(True)
    grad = torch.autograd.grad(F.cross_entropy(model(x), y), x)[0]
    return (x + eps * grad.sign()).clamp(-1, 1).detach()

def pgd(model, x, y, eps, steps=10):
    alpha = 2.5 * eps / steps
    x0 = x.clone().detach()
    xa = (x0 + torch.empty_like(x0).uniform_(-eps, eps)).clamp(-1, 1).detach()
    for _ in range(steps):
        xa.requires_grad_(True)
        grad = torch.autograd.grad(F.cross_entropy(model(xa), y), xa)[0]
        xa = xa.detach() + alpha * grad.sign()
        xa = (x0 + (xa - x0).clamp(-eps, eps)).clamp(-1, 1).detach()
    return xa

## 9. Flat ASR vs RW-ASR

Only objects the model gets right on clean images are scored, for both metrics.

In [ ]:
@torch.no_grad()
def predict(model, x):
    return model(x).argmax(1)

def run_attack(model, dl, eps, attack="pgd"):
    model.eval()
    ys, cleans, advs, ws = [], [], [], []
    for x, y, w in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        clean = predict(model, x)
        xa = fgsm(model, x, y, eps) if attack == "fgsm" else pgd(model, x, y, eps)
        ys.append(y.cpu().numpy()); cleans.append(clean.cpu().numpy())
        advs.append(predict(model, xa).cpu().numpy()); ws.append(w.numpy())
    return tuple(np.concatenate(a) for a in (ys, cleans, advs, ws))

def score(y, clean, adv, w):
    ok = clean == y
    y, adv, w = y[ok], adv[ok], w[ok]
    failed = adv != y
    out = dict(
        n_attackable=int(len(y)),
        flat_asr=float(failed.mean()),
        rw_asr=float(w[failed].sum() / w.sum()) if w.sum() > 0 else 0.0,
        expected_fatalities_per_1k=float(1000 * w[failed].sum() / len(y)),
    )
    for i, c in IDX_TO_CLASS.items():
        if (y == i).any():
            out[f"asr_{c}"] = float(failed[y == i].mean())
    return out

def evaluate_grid(epsilons):
    rows, raw = [], {}
    for mname, m in MODELS.items():
        for atk in ("fgsm", "pgd"):
            for eps in epsilons:
                y, cl, ad, w = run_attack(m, test_dl, eps, atk)
                raw[(mname, atk, eps)] = (y, cl, ad, w)
                r = score(y, cl, ad, w)
                r.update(model=mname, attack=atk, eps=eps, eps_255=round(eps_over_255(eps), 1))
                rows.append(r)
    return pd.DataFrame(rows), raw

In [ ]:
# first sweep (most runs hit ASR ~1.0, kept for reference)
results, RAW = evaluate_grid(EPSILONS)
results.to_csv(RESDIR / "grid_results.csv", index=False)

# main sweep
results_fine, RAW_FINE = evaluate_grid(EPSILONS_FINE)
results_fine.to_csv(RESDIR / "grid_results_fine.csv", index=False)
results_fine[["model", "attack", "eps", "flat_asr", "rw_asr"]].round(4)

### Road users only

Background patches have weight 0 and are rarely flipped, so they pull flat ASR down.
This compares RW-ASR against flat ASR computed without them.

In [ ]:
road = CLASS_TO_IDX["road scene"]
rows = []
for (mname, atk, eps), (y, cl, ad, w) in RAW_FINE.items():
    ok = cl == y
    m = ok & (y != road)
    rows.append(dict(model=mname, attack=atk, eps=eps,
                     flat_all=(ad[ok] != y[ok]).mean(),
                     flat_road_users=(ad[m] != y[m]).mean(),
                     rw_asr=score(y, cl, ad, w)["rw_asr"]))
road_users = pd.DataFrame(rows)
road_users.to_csv(RESDIR / "road_users_only.csv", index=False)
road_users.round(4)

## 10. Figures

In [ ]:
OP_EPS = 0.008  # cnn PGD flat ASR is ~0.45 here, nothing saturated

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for j, atk in enumerate(["fgsm", "pgd"]):
    for mname, mk, ls in [("cnn", "o", "-"), ("resnet18", "s", "--")]:
        s = results_fine[(results_fine.model == mname) & (results_fine.attack == atk)].sort_values("eps")
        ax[j].plot(s.eps, s.flat_asr, ls, marker=mk, color="#1f3864", label=f"{mname} flat ASR")
        ax[j].plot(s.eps, s.rw_asr, ls, marker=mk, color="#c0392b", label=f"{mname} RW-ASR")
    ax[j].set_title(atk.upper())
    ax[j].set_xlabel("epsilon (normalized, [-1,1] space)")
ax[0].set_ylabel("attack success rate")
ax[0].legend(fontsize=8, loc="upper left")
plt.tight_layout(); savefig("fig5_ranking"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
cols = {"pedestrian": "#c0392b", "cyclist": "#d68910", "car": "#1f3864", "non-road object": "#7f8c8d"}
for j, mname in enumerate(["cnn", "resnet18"]):
    s = results_fine[(results_fine.model == mname) & (results_fine.attack == "pgd")].sort_values("eps")
    for c, col in cols.items():
        ax[j].plot(s.eps, s[f"asr_{c}"], "-o", color=col, label=c)
    ax[j].set_title(f"{mname} (PGD)"); ax[j].set_xlabel("epsilon")
ax[0].set_ylabel("per-class attack success rate")
ax[0].legend(fontsize=9)
plt.tight_layout(); savefig("fig6_per_class"); plt.show()

### Sensitivity

Re-scores the saved predictions with different assumptions (no re-attacking).

In [ ]:
def rescore(key, cfg=None, severity=None):
    y, cl, ad, _ = RAW_FINE[key]
    w = np.array([float(risk_weight(c, z, cfg=cfg, severity=severity))
                  for c, z in zip(test_cls, test_z)], dtype=np.float32)
    return score(y, cl, ad, w)

rows = []
for v in [3.0, 5.0, 7.0, 9.0, 11.0, 13.0, 15.0]:
    sev = dict(SEVERITY, pedestrian=float(v))
    for m in MODELS:
        rows.append(dict(param="pedestrian severity weight", value=v, model=m,
                         rw_asr=rescore((m, "pgd", OP_EPS), severity=sev)["rw_asr"]))
for v in [25, 35, 45]:
    for m in MODELS:
        rows.append(dict(param="ego speed (mph)", value=v, model=m,
                         rw_asr=rescore((m, "pgd", OP_EPS), cfg=dict(CFG, ego_speed_mph=float(v)))["rw_asr"]))
for v in [0.2, 0.4, 0.6, 0.8]:
    for m in MODELS:
        rows.append(dict(param="failure delay (s)", value=v, model=m,
                         rw_asr=rescore((m, "pgd", OP_EPS), cfg=dict(CFG, delay_fail=float(v)))["rw_asr"]))
sweep = pd.DataFrame(rows)
sweep.to_csv(RESDIR / "sensitivity_fine.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, p in zip(ax, sweep.param.unique()):
    s = sweep[sweep.param == p]
    for m, col in [("cnn", "#1f3864"), ("resnet18", "#c0392b")]:
        ss = s[s.model == m].sort_values("value")
        a.plot(ss.value, ss.rw_asr, "-o", color=col, label=m)
    a.set_xlabel(p); a.set_ylabel("RW-ASR"); a.set_ylim(0, 1)
ax[0].legend()
plt.suptitle(f"RW-ASR under different assumptions (PGD, eps={OP_EPS})", y=1.04)
plt.tight_layout(); savefig("fig7_sensitivity"); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y, cl, ad, _ = RAW_FINE[("cnn", "pgd", OP_EPS)]
names = [IDX_TO_CLASS[i] for i in range(len(CLASSES))]

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (preds, title) in zip(ax, [(cl, "Clean"), (ad, f"PGD (eps={OP_EPS})")]):
    cm = confusion_matrix(y, preds, labels=range(len(CLASSES)))
    cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    a.set_xticks(range(len(names))); a.set_xticklabels(names, rotation=40, ha="right")
    a.set_yticks(range(len(names))); a.set_yticklabels(names)
    a.set_title(f"{title} - baseline CNN"); a.set_xlabel("predicted"); a.grid(False)
    for r in range(len(names)):
        for c in range(len(names)):
            if cm[r, c]:
                a.text(c, r, cm[r, c], ha="center", va="center", fontsize=8,
                       color="white" if cmn[r, c] > 0.5 else "black")
ax[0].set_ylabel("true label")
plt.tight_layout(); savefig("fig8_confusion"); plt.show()

for mname in MODELS:
    yy, cc, _, _ = RAW_FINE[(mname, "pgd", OP_EPS)]
    print(f"--- {mname} clean ---")
    print(classification_report(yy, cc, labels=range(len(CLASSES)), target_names=names, digits=3))

In [ ]:
def denorm(t):
    return (t.detach().cpu() * NORM_STD + NORM_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()

ped = CLASS_TO_IDX["pedestrian"]
yv, clv, adv_, _ = RAW_FINE[("cnn", "pgd", OP_EPS)]
cand = np.where((yv == ped) & (clv == ped) & (adv_ != ped))[0]
Xte = X_all[te_mask]

fig, ax = plt.subplots(3, 4, figsize=(11, 8.7))
for r, idx in enumerate(cand[:3]):
    x = ((Xte[idx].float() / 255.0 - NORM_MEAN) / NORM_STD).unsqueeze(0).to(DEVICE)
    yt = torch.tensor([int(yv[idx])], device=DEVICE)
    xf, xp = fgsm(cnn, x, yt, OP_EPS), pgd(cnn, x, yt, OP_EPS)
    diff = (xp - x)[0]
    diff = (diff - diff.min()) / (diff.max() - diff.min() + 1e-9)
    panels = [(denorm(x[0]), f"clean\npred: {IDX_TO_CLASS[int(clv[idx])]}"),
              (denorm(xf[0]), f"FGSM\npred: {IDX_TO_CLASS[int(predict(cnn, xf)[0])]}"),
              (denorm(xp[0]), f"PGD\npred: {IDX_TO_CLASS[int(predict(cnn, xp)[0])]}"),
              (diff.cpu().permute(1, 2, 0).numpy(), "PGD perturbation\n(amplified)")]
    for c, (img, title) in enumerate(panels):
        ax[r, c].imshow(img); ax[r, c].axis("off"); ax[r, c].set_title(title, fontsize=10)
plt.suptitle(f"Pedestrian crops flipped by a {eps_over_255(OP_EPS):.2f}/255 perturbation", y=1.0)
plt.tight_layout(); savefig("fig9_adversarial_examples"); plt.show()

## 11. Random noise control

Same eps, uniform random noise instead of gradients. Checks whether the attacks are
targeted or if any small change would flip the labels.

In [ ]:
@torch.no_grad()
def run_noise(model, dl, eps):
    model.eval()
    ys, cleans, advs, ws = [], [], [], []
    for x, y, w in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        xn = (x + torch.empty_like(x).uniform_(-eps, eps)).clamp(-1, 1)
        ys.append(y.cpu().numpy()); cleans.append(predict(model, x).cpu().numpy())
        advs.append(predict(model, xn).cpu().numpy()); ws.append(w.numpy())
    return tuple(np.concatenate(a) for a in (ys, cleans, advs, ws))

def gap_pct(rw, flat):
    return 100 * (rw - flat) / flat if flat > 0 else 0.0

rows = []
for mname, m in MODELS.items():
    for eps in EPSILONS_FINE:
        rn = score(*run_noise(m, test_dl, eps))
        ra = score(*run_attack(m, test_dl, eps, "pgd"))
        rows.append(dict(model=mname, eps=eps,
                         noise_flat=rn["flat_asr"], noise_rw=rn["rw_asr"],
                         noise_gap_pct=gap_pct(rn["rw_asr"], rn["flat_asr"]),
                         pgd_flat=ra["flat_asr"], pgd_rw=ra["rw_asr"],
                         pgd_gap_pct=gap_pct(ra["rw_asr"], ra["flat_asr"])))
noise = pd.DataFrame(rows)
noise.to_csv(RESDIR / "noise_control.csv", index=False)
noise.round(4)

## 12. Save outputs

In [ ]:
shutil.make_archive("rwasr_figures", "zip", ".", "figures")
shutil.make_archive("rwasr_results", "zip", ".", "results")